In [10]:
import pandas as pd
import requests
from io import StringIO #questo è importante per la questione del dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler



In [11]:
#importare i dati sul territorio
comuni = pd.read_csv("Comuni - Dimensione Data Indagine 31-12-2020 Stampa 19062026105027.csv", sep = ";")

In [12]:
comuni.shape

(7903, 17)

In [13]:
comuni.columns

Index(['Codice Ripartizione geografica', 'Codice Regione',
       'Codice Provincia (Storico)', 'Codice Provincia/Uts',
       'Codice Comune (alfanumerico)', 'Codice Comune (numerico)', 'Comune',
       'Comune (dizione straniera)', 'Sigla automobilistica',
       'Capoluogo di Provincia/Uts', 'Capoluogo di Regione',
       'Popolazione legale', 'Anno Censimento', 'Superficie (Kmq)',
       'Anno (Superficie)', 'Popolazione residente',
       'Anno (Popolazione residente)'],
      dtype='str')

In [14]:
#importare i dati sugli incidenti (URL API)
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
#bisogna scaricare i dati come CSV quindi:
dati_incidenti=requests.get(url, headers={"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"})
# usando StringIO evitiamo di salvare il CSV sul disco e permettiamo a pandas di leggerlo 
incidenti = pd.read_csv(StringIO(dati_incidenti.text))



In [15]:
#esploro i dati
print(incidenti.columns.tolist())
print(incidenti.head())
print(incidenti.shape)

['Error during writing responce']
Empty DataFrame
Columns: [Error during writing responce]
Index: []
(0, 1)


In [ ]:
#visto che ci sono diversi NaN eliminiamoli e verifichiamo che siano stati effettivamente eliminati facendo ancora totlist
incidenti = incidenti.dropna(axis=1, how="all")
print(incidenti.columns.tolist())

In [ ]:
print(incidenti["REF_AREA"].unique()[:50])
print(incidenti["REF_AREA"].dtype)

In [ ]:
print(incidenti["REF_AREA"].nunique())

In [ ]:
print(comuni.columns.tolist())
print(comuni.head())
#ora so che i codici del dataset precedente corrispondono ai comuni quindi so dove joinare

In [ ]:
#facciamo la join
df = incidenti.merge(comuni,
left_on="REF_AREA",
right_on="Codice Comune (numerico)",
how="left")
#verifichiamo che funziona
print(df[["REF_AREA", "Comune"]].head())

In [ ]:
#ora faccio un df con solo gli incidenti stradali, killinj sta per morti e feriti ma ddobbiamo concentrarci sugli INCIDENTI
incidenti_stradali = df[df["DATA_TYPE"] == "ROADACC"]

In [ ]:
#vediamo quanti incidenti per mille abitanti ci sono facendo una divisione tra gli incidenti (obs value) e la popolazione
incidenti_stradali["incidenti_per_mille_abitanti"] = (
incidenti_stradali["OBS_VALUE"] /
incidenti_stradali["Popolazione residente"] * 1000)
#ho moltiplicato per mille per renerlo più intuitivo altrimenti non si avrebbe avuto la percezione effettiva del numero degli incidenti

In [ ]:
#vediamo se ci sono outliers con un grafico mettendo limite 100
sns.boxplot(y=incidenti_stradali["OBS_VALUE"])
plt.ylim(0, 100)
plt.show()
#già si capisce che ci sono molti ouliers probabilmente perchè in città piu grandi ci sono piu incidenti, dobbiamo tenerli in considerazione 
#perchè non è un errore ma sono dati reali

In [ ]:
#vediamo quali comuni hanno piu incidenti (gruppando comuni) e ordiniamo per valori decrescenti
top10 = incidenti_stradali.groupby("Comune")["OBS_VALUE"].sum()
top10 = top10.sort_values(ascending=False)
print(top10.head(10))
#come ci si poteva aspettare le città più popolose presentano più incidenti

In [ ]:
#preparo i dati per la regressione
dati_regressione = incidenti_stradali.dropna(subset=["Popolazione residente", "OBS_VALUE"])#togliamo eventuali NaN
X = dati_regressione[["Popolazione residente"]]
y = dati_regressione["OBS_VALUE"]

In [ ]:
#faccio il modello
modello = LinearRegression()
modello.fit(X, y)

In [ ]:
#calcoliamo R^2
r2 = modello.score(X, y)
print("R²:", r2)
#86%! si vede che il numero della popolazione spiega il numero di incidenti

In [ ]:
#ora facciamo il grafico
plt.scatter(dati_regressione["Popolazione residente"],
dati_regressione["OBS_VALUE"],alpha=0.5)
y_pred = modello.predict(X)#predict degli incidenti in base alla popolazione
plt.plot(dati_regressione["Popolazione residente"],y_pred,color="red")
plt.xlabel("Popolazione residente")
plt.ylabel("Numero di incidenti")
plt.title("Regressione lineare")
plt.show()

In [ ]:
#p value ecc.
X_stats = sm.add_constant(X)
modello_stats = sm.OLS(y, X_stats)
risultati = modello_stats.fit()
print(risultati.summary())

In [ ]:
#nella cella sotto stavo creando una nuova variabile ma un valore non tornava quindi converto a float
#bisogna convertire in float perchè c'era un numero anomalo
#convertiamo la colonna in stringa
incidenti_stradali["Superficie (Kmq)"] = incidenti_stradali["Superficie (Kmq)"].astype(str)
#eliminiamo il punto usato come separatore delle migliaia
incidenti_stradali["Superficie (Kmq)"] = incidenti_stradali["Superficie (Kmq)"].str.replace(".", "", regex=False)
#sostituiamo la virgola con il punto decimale
incidenti_stradali["Superficie (Kmq)"] = incidenti_stradali["Superficie (Kmq)"].str.replace(",", ".", regex=False)
#convertiamo la colonna in float
incidenti_stradali["Superficie (Kmq)"] = incidenti_stradali["Superficie (Kmq)"].astype(float)

In [ ]:
#crao la variabile incidendi per kmq
incidenti_stradali["incidenti_per_kmq"] = (
incidenti_stradali["OBS_VALUE"] /
incidenti_stradali["Superficie (Kmq)"])

In [ ]:
print(incidenti_stradali["Superficie (Kmq)"].dtype)

In [ ]:
# Eliminiamo le righe che non hanno valori validi
# per poter fare il clustering
incidenti_stradali = incidenti_stradali.dropna(subset=["incidenti_per_mille_abitanti", "incidenti_per_kmq"])

In [ ]:
#seleziono le variabili
X = incidenti_stradali[["incidenti_per_mille_abitanti","incidenti_per_kmq"]]

In [ ]:
#normalizzo i dati
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X)

In [ ]:
#k means
kmeans = KMeans(n_clusters=3, random_state=42)
cluster = kmeans.fit_predict(X_norm)

In [ ]:
#aggiungiamo il cluster al df
incidenti_stradali["cluster"] = cluster

In [ ]:
sns.scatterplot(data=incidenti_stradali,
x="incidenti_per_mille_abitanti",
y="incidenti_per_kmq",
hue="cluster")

plt.xlabel("Incidenti per 1000 abitanti")
plt.ylabel("Incidenti per kmq")
plt.title("Clustering dei comuni")
plt.show()

In [ ]:
#devo usare la virgola come separatore decimale altrimenti power bi non lo legge bene e restituisce numeri assurdi
incidenti_stradali.to_csv("incidenti_stradali_finale.csv",sep=";",decimal=",",index=False)